# Oglądam rozkłady chromVAR scrorów w grupach 1-1 i other (21.02.26)
Idea jest taka, żeby włączyć do klasyfikatora więcej informacji z rozkładów niż tylko różnica pomiędzy średnimi.

In [ ]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from utils import *

In [ ]:
windows = ['06-08', '10-12', '14-16']
loop_ids = ['L222', 'L400']
motif_ids = ['M0596-1.02', 'M6277-1.02']

In [ ]:
def distributions(loop_ids: list[str], motif_ids: list[str], loops_df: pd.DataFrame, motifs_df: pd.DataFrame) -> dict[dict]:
    """
    Returns whole z-score distributions for all activity profiles (lists of values)
    Args:
        loop_ids (list[str]): Loop indentifiers (like 'L417')
        motif_ids (list[str]): motif indentifiers (like 'M0111-1.02')
        loops_df, motifs_df (pd.DataFrame): dataframes loaded by load_window()

    Returns:
        result (dict): a dictionary with 
    """
    result = {loop: {motif: {"1-1": [], "1-0": [], "0-1": [], "0-0": []} for motif in motif_ids} for loop in loop_ids}
    for loop_id in loop_ids:
        loop_values = loops_df.loc[loop_id]
        cells_11 = loop_values[loop_values == 11].index
        cells_10 = loop_values[loop_values == 10].index
        cells_01 = loop_values[loop_values == 1].index
        cells_00 = loop_values[loop_values == 0].index

        for motif_id in motif_ids:
            motif_values = motifs_df.loc[motif_id]

            result[loop_id][motif_id]["1-1"] = motif_values.loc[cells_11] if len(cells_11) > 0 else np.nan
            result[loop_id][motif_id]["1-0"] = motif_values.loc[cells_10] if len(cells_10) > 0 else np.nan
            result[loop_id][motif_id]["0-1"] = motif_values.loc[cells_01] if len(cells_01) > 0 else np.nan
            result[loop_id][motif_id]["0-0"] = motif_values.loc[cells_00] if len(cells_00) > 0 else np.nan
            
    return result

In [ ]:
def plot_distributions(window, loop_id, motif_id, motif_name, big_dict: dict, save=False) -> None:
    """
    Plots four distributions ('1-1', '1-0', '0-1', '0-0')
    as faceted histograms (one under the other),
    each with its own y-scale.
    """

    lowest_dict = big_dict[window][loop_id][motif_id]

    colors = {
        "1-1": "#1f77b4",
        "1-0": "#ff7f0e",
        "0-1": "#2ca02c",
        "0-0": "#d62728",
    }

    labels = ["1-1", "1-0", "0-1", "0-0"]

    fig, axes = plt.subplots(
        nrows=4,
        ncols=1,
        figsize=(8, 12),
        sharex=True
    )

    for ax, label in zip(axes, labels):
        values = lowest_dict.get(label)

        if isinstance(values, pd.Series) and len(values) > 0:
            mean_val = values.mean()

            sns.histplot(
                values,
                bins=100,
                stat="count",
                color=colors[label],
                alpha=0.85,
                edgecolor="black",
                ax=ax
            )

            # Mean vertical line
            ax.axvline(
                mean_val,
                color="black",
                linestyle="--",
                linewidth=1.0
            )

            # Mean annotation
            ax.text(
                0.02, 0.95,
                f"mean = {mean_val:.3f}",
                transform=ax.transAxes,
                verticalalignment="top",
                horizontalalignment="left",
                fontsize=15,
                color = 'black',
                bbox=dict(
                    facecolor="white",
                    edgecolor="none",
                    alpha=0.1
                )
            )

            ax.set_ylabel(f"{label}\n(n={len(values)})")

        else:
            ax.text(
                0.5, 0.5, "No data",
                ha='center', va='center',
                transform=ax.transAxes
            )
            ax.set_ylabel(label)

        ax.set_xlim(-4, 4)
        ax.set_xlabel("ChromVAR score")
        ax.grid(alpha=0.3)

    axes[0].set_title(
        f"ChromVAR score distributions for\n"
        f'motif: {motif_id} - "{motif_name}",\nloop: {loop_id}, \nwindow: hrs{window}'
    )

    plt.tight_layout()


    if save:
        dir_path = "results/figures/chromvar_distributions"
        os.makedirs(dir_path, exist_ok=True)
        path = os.path.join(dir_path, f"{loop_id}_{motif_id}_{window}.png")
        plt.savefig(path, dpi=300)
    else:
        plt.show()

In [ ]:
big_dict = {w: {} for w in windows}
for window in windows:
    big_dict[window] = distributions(loop_ids, motif_ids, *load_window(window=window))

motif_lookup = pd.read_csv("data/motif_names.tsv", sep="\t")

for loop_id in loop_ids:
    for motif_id in motif_ids:
        motif_name = motif_lookup[motif_lookup["id"] == motif_id]["name"]
        motif_name = str(list(motif_name)[0])
        for window in windows:
            plot_distributions(window=window, loop_id=loop_id, motif_id=motif_id, motif_name=motif_name, big_dict=big_dict, save=True)